In [29]:
import os
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    HubertModel, Wav2Vec2FeatureExtractor,
    AutoTokenizer, AutoModel
)
from datasets import load_from_disk
import librosa
from tqdm import tqdm

In [30]:
class CFG:
    # models
    audio_model_name  = "facebook/hubert-base-ls960"
    text_model_name   = "sentence-transformers/all-MiniLM-L6-v2"

    # embedding dims
    audio_hidden_dim  = 768   # HuBERT-base output
    proj_dim          = 384   # match MiniLM output dim

    # audio
    sample_rate       = 16000
    max_audio_len     = 5     # seconds — adjust to your clip lengths

    # training
    batch_size        = 32
    num_epochs        = 10
    lr                = 3e-5
    warmup_steps      = 500
    temperature       = 0.07  # InfoNCE temperature

    # loss weights
    lambda_cosine     = 1.0
    lambda_infonce    = 1.0

    # misc
    seed              = 42
    num_workers       = 2
    device            = "cuda" if torch.cuda.is_available() else "cpu"
    save_path         = "audio_encoder_distilled.pt"

    # set this to your dataset path on Kaggle
    dataset_path      = "/kaggle/input/ms-marco-audio-100k-no-prefix/msmarco_queries_tts"

In [31]:
def mean_pool(token_embeddings, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)


@torch.no_grad()
def precompute_text_embeddings(dataset, tokenizer, text_model, device, batch_size=256, cache_path="text_embeddings.npy"):
    if os.path.exists(cache_path):
        print(f"Loading cached text embeddings from {cache_path}")
        return np.load(cache_path)

    print("Pre-computing text embeddings...")
    text_model.eval().to(device)
    all_embeddings = []

    queries = dataset["query"]
    for i in tqdm(range(0, len(queries), batch_size)):
        batch_queries = queries[i : i + batch_size]
        encoded = tokenizer(batch_queries, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        output = text_model(**encoded)
        emb = mean_pool(output.last_hidden_state, encoded["attention_mask"])
        emb = F.normalize(emb, dim=-1)
        all_embeddings.append(emb.cpu().numpy())

    all_embeddings = np.concatenate(all_embeddings, axis=0)
    np.save(cache_path, all_embeddings)
    print(f"Saved text embeddings to {cache_path} — shape: {all_embeddings.shape}")
    return all_embeddings

In [32]:
class AudioTextDataset(Dataset):
    def __init__(self, dataset, text_embeddings, feature_extractor, cfg):
        self.dataset           = dataset
        self.text_embeddings   = text_embeddings  # numpy array [N, 384]
        self.feature_extractor = feature_extractor
        self.cfg               = cfg
        self.max_samples       = cfg.sample_rate * cfg.max_audio_len

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset[idx]

        # load audio
        base_dir = "/kaggle/input/ms-marco-audio-100k-no-prefix/msmarco_query_audio"
        fname = Path(row["audio_path"].replace("\\","/")).name
        audio, sr = librosa.load(os.path.join(base_dir, fname), sr=self.cfg.sample_rate, mono=True)

        # pad or truncate
        if len(audio) > self.max_samples:
            audio = audio[:self.max_samples]
        else:
            audio = np.pad(audio, (0, self.max_samples - len(audio)))

        # feature extractor handles normalization
        inputs = self.feature_extractor(
            audio,
            sampling_rate=self.cfg.sample_rate,
            return_tensors="pt",
            padding=False,
        )
        input_values = inputs.input_values.squeeze(0)  # [T]

        text_emb = torch.tensor(self.text_embeddings[idx], dtype=torch.float32)  # [384]

        return input_values, text_emb


def collate_fn(batch):
    input_values, text_embs = zip(*batch)
    # pad audio sequences to the same length in the batch
    max_len = max(x.shape[0] for x in input_values)
    padded = torch.zeros(len(input_values), max_len)
    for i, iv in enumerate(input_values):
        padded[i, :iv.shape[0]] = iv
    text_embs = torch.stack(text_embs, dim=0)
    return padded, text_embs

In [33]:
class AudioEncoderWithProjection(nn.Module):
    """
    HuBERT-base + mean pooling + projection head to 384-dim.
    Only the projection head + last N transformer layers are trained
    to stay within Kaggle memory limits.
    """
    def __init__(self, cfg, unfreeze_last_n_layers=4):
        super().__init__()
        self.hubert = HubertModel.from_pretrained(cfg.audio_model_name)

        # freeze everything first
        for param in self.hubert.parameters():
            param.requires_grad = False

        # unfreeze the last N encoder layers
        encoder_layers = self.hubert.encoder.layers
        for layer in encoder_layers[-unfreeze_last_n_layers:]:
            for param in layer.parameters():
                param.requires_grad = True

        # projection head: 768 → 384 with a bottleneck
        self.projection = nn.Sequential(
            nn.Linear(cfg.audio_hidden_dim, cfg.audio_hidden_dim),
            nn.GELU(),
            nn.LayerNorm(cfg.audio_hidden_dim),
            nn.Linear(cfg.audio_hidden_dim, cfg.proj_dim),
        )

    def forward(self, input_values):
        # HuBERT outputs [B, T', 768]
        outputs = self.hubert(input_values=input_values)
        hidden  = outputs.last_hidden_state          # [B, T', 768]
        pooled  = hidden.mean(dim=1)                 # [B, 768]
        proj    = self.projection(pooled)            # [B, 384]
        return F.normalize(proj, dim=-1)             # L2 normalize

In [34]:
class DistillationLoss(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.temperature     = cfg.temperature
        self.lambda_cosine   = cfg.lambda_cosine
        self.lambda_infonce  = cfg.lambda_infonce

    def cosine_loss(self, audio_emb, text_emb):
        # 1 - cosine_similarity, averaged over batch
        return 1.0 - F.cosine_similarity(audio_emb, text_emb, dim=-1).mean()

    def infonce_loss(self, audio_emb, text_emb):
        """
        Symmetric InfoNCE (like CLIP).
        Each audio embedding should be closest to its paired text embedding.
        Both audio_emb and text_emb are already L2-normalized.
        """
        B = audio_emb.size(0)
        logits = torch.matmul(audio_emb, text_emb.T) / self.temperature  # [B, B]
        labels = torch.arange(B, device=audio_emb.device)
        loss_a = F.cross_entropy(logits, labels)       # audio → text
        loss_t = F.cross_entropy(logits.T, labels)     # text → audio
        return (loss_a + loss_t) / 2.0

    def forward(self, audio_emb, text_emb):
        cosine  = self.cosine_loss(audio_emb, text_emb)
        infonce = self.infonce_loss(audio_emb, text_emb)
        total   = self.lambda_cosine * cosine + self.lambda_infonce * infonce
        return total, cosine.item(), infonce.item()

In [35]:
def train(cfg):
    torch.manual_seed(cfg.seed)

    # ── load dataset ──
    print("Loading dataset...")
    dataset = load_from_disk(cfg.dataset_path)

    # ── text teacher (frozen, only used for pre-computation) ──
    print("Loading text model...")
    tokenizer  = AutoTokenizer.from_pretrained(cfg.text_model_name)
    text_model = AutoModel.from_pretrained(cfg.text_model_name)
    text_model.eval()

    text_embeddings = precompute_text_embeddings(
        dataset, tokenizer, text_model, cfg.device,
        batch_size=256, cache_path="text_embeddings.npy"
    )
    del text_model  # free memory after pre-computation

    # ── audio feature extractor + dataset ──
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(cfg.audio_model_name)
    train_dataset     = AudioTextDataset(dataset, text_embeddings, feature_extractor, cfg)
    train_loader      = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        collate_fn=collate_fn,
        pin_memory=True,
    )

    # ── model ──
    print("Loading audio model...")
    model = AudioEncoderWithProjection(cfg, unfreeze_last_n_layers=4).to(cfg.device)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable:,}")

    # ── optimizer & scheduler ──
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg.lr, weight_decay=1e-2
    )
    total_steps = len(train_loader) * cfg.num_epochs
    scheduler   = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=cfg.lr,
        total_steps=total_steps,
        pct_start=cfg.warmup_steps / total_steps,
    )

    criterion = DistillationLoss(cfg)
    scaler    = torch.cuda.amp.GradScaler()  # mixed precision

    # ── train ──
    best_loss = float("inf")
    for epoch in range(cfg.num_epochs):
        model.train()
        running_loss = running_cos = running_nce = 0.0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.num_epochs}")
        for step, (input_values, text_emb) in enumerate(pbar):
            input_values = input_values.to(cfg.device)
            text_emb     = text_emb.to(cfg.device)

            with torch.cuda.amp.autocast():
                audio_emb              = model(input_values)
                loss, cos_l, nce_l     = criterion(audio_emb, text_emb)

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running_loss += loss.item()
            running_cos  += cos_l
            running_nce  += nce_l

            if step % 50 == 0:
                pbar.set_postfix({
                    "loss": f"{loss.item():.4f}",
                    "cos":  f"{cos_l:.4f}",
                    "nce":  f"{nce_l:.4f}",
                    "lr":   f"{scheduler.get_last_lr()[0]:.2e}",
                })

        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch+1} — avg loss: {avg_loss:.4f} | cos: {running_cos/len(train_loader):.4f} | nce: {running_nce/len(train_loader):.4f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save({
                "epoch":       epoch + 1,
                "model_state": model.state_dict(),
                "loss":        best_loss,
                "cfg":         cfg.__dict__,
            }, cfg.save_path)
            print(f"  ✓ Saved best model (loss={best_loss:.4f})")

    print("Training complete.")
    return model

In [36]:
if __name__ == "__main__":
    cfg = CFG()
    print(f"Device: {cfg.device}")
    trained_model = train(cfg)

Device: cuda
Loading dataset...
Loading text model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading cached text embeddings from text_embeddings.npy
Loading audio model...


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

...

Epoch 8 — avg loss: 0.6929 | cos: 0.4984 | nce: 0.1946
  ✓ Saved best model (loss=0.6929)


Epoch 9/10: 100%|██████████| 3157/3157 [41:52<00:00,  1.26it/s, loss=0.5863, cos=0.4558, nce=0.1305, lr=7.60e-07]


Epoch 9 — avg loss: 0.6732 | cos: 0.4922 | nce: 0.1811
  ✓ Saved best model (loss=0.6732)


Epoch 10/10: 100%|██████████| 3157/3157 [41:49<00:00,  1.26it/s, loss=0.6970, cos=0.5240, nce=0.1730, lr=1.22e-10]

Epoch 10 — avg loss: 0.6814 | cos: 0.4931 | nce: 0.1883
Training complete.
